# 01 — House index audit

## Objectif

Télécharger et parser les index XML House 2013–2026.

Produire :
- `house_filings_index.csv` ;
- `house_ptr_index.csv` ;
- `ptr_index_YYYY.csv` ;
- `reports/house_index_audit.md`.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.house_index import (
    build_house_filings_index, filter_ptr_filings, save_index_outputs,
    summarize_house_index, write_house_index_report, build_house_zip_url,
    build_house_pdf_url,
)
from src.utils import PROCESSED_HOUSE_DIR, AUDIT_DIR

print("Imports OK")

Imports OK


In [2]:
START_YEAR = 2013
END_YEAR = 2026
SLEEP_SECONDS = 0.25
FORCE_DOWNLOAD = False

print(build_house_zip_url(START_YEAR))
print(build_house_pdf_url(2026, "20034201"))

https://disclosures-clerk.house.gov/public_disc/financial-pdfs/2013FD.zip
https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2026/20034201.pdf


## Exécution

Les anciens chiffres sont des repères d’audit. Ils ne sont pas hardcodés.

In [3]:
df_all, logs = build_house_filings_index(
    start_year=START_YEAR,
    end_year=END_YEAR,
    sleep_seconds=SLEEP_SECONDS,
    force=FORCE_DOWNLOAD,
)

print("df_all shape:", df_all.shape)
logs

House years:   0%|          | 0/14 [00:00<?, ?it/s]

df_all shape: (37445, 11)


,year,url,path,status,http_status,error,xml_path,n_filings
0,2013,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...,ok,200,,/Users/lemairealice/Downloads/Jupiter/congress...,2209
1,2014,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...,ok,200,,/Users/lemairealice/Downloads/Jupiter/congress...,2788
2,2015,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...,ok,200,,/Users/lemairealice/Downloads/Jupiter/congress...,2297
3,2016,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...,ok,200,,/Users/lemairealice/Downloads/Jupiter/congress...,2704
4,2017,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...,ok,200,,/Users/lemairealice/Downloads/Jupiter/congress...,3488
5,2018,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...,ok,200,,/Users/lemairealice/Downloads/Jupiter/congress...,3443
6,2019,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...,ok,200,,/Users/lemairealice/Downloads/Jupiter/congress...,3636
7,2020,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...,ok,200,,/Users/lemairealice/Downloads/Jupiter/congress...,2930
8,2021,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...,ok,200,,/Users/lemairealice/Downloads/Jupiter/congress...,2718
9,2022,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...,ok,200,,/Users/lemairealice/Downloads/Jupiter/congress...,2743


In [4]:
df_ptr = filter_ptr_filings(df_all)
print("df_ptr shape:", df_ptr.shape)

if not df_all.empty:
    display(df_all.head())
if not df_ptr.empty:
    display(df_ptr.head())

df_ptr shape: (8248, 11)


,year,filing_type,filing_date_raw,filing_date,doc_id,last_name,first_name,declarant_name_raw,state_dst,pdf_url,source_xml_path
0,2013,X,1/28/2013,2013-01-28,8209738,ACKERMAN,GARY,GARY ACKERMAN,NY05,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...
1,2013,T,3/14/2013,2013-03-14,8209998,ACKERMAN,GARY,GARY ACKERMAN,NY05,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...
2,2013,O,8/1/2013,2013-08-01,8212748,ADAMS,ALMA SHEALEY,ALMA SHEALEY ADAMS,NC12,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...
3,2013,O,5/15/2013,2013-05-15,8211641,ADAMS,"SANDRA ""SANDY""","SANDRA ""SANDY"" ADAMS",FL24,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...
4,2013,T,1/17/2013,2013-01-17,8209668,ADAMS,"SANDRA ""SANDY""","SANDRA ""SANDY"" ADAMS",FL24,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...


,year,filing_type,filing_date_raw,filing_date,doc_id,last_name,first_name,declarant_name_raw,state_dst,pdf_url,source_xml_path
0,2013,P,4/9/2014,2014-04-09,8214458,Black,Diane,Diane Black,TN06,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...
1,2013,P,4/7/2014,2014-04-07,8214460,Collins,Chris,Chris Collins,NY27,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...
2,2013,P,12/3/2014,2014-12-03,9105684,Grayson,Alan,Alan Grayson,FL09,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...
3,2013,P,12/3/2014,2014-12-03,9105686,Grayson,Alan,Alan Grayson,FL09,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...
4,2013,P,12/3/2014,2014-12-03,9105705,Grayson,Alan,Alan Grayson,FL09,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...


In [5]:
summary = summarize_house_index(df_all, df_ptr, logs)
summary

{'timestamp': '2026-06-23T15:56:59+00:00',
 'years_ok': 14,
 'n_total_filings': 37445,
 'n_total_ptr': 8248,
 'ptr_by_year': {'2013': 8,
  '2014': 708,
  '2015': 728,
  '2016': 765,
  '2017': 801,
  '2018': 830,
  '2019': 683,
  '2020': 733,
  '2021': 680,
  '2022': 624,
  '2023': 460,
  '2024': 451,
  '2025': 515,
  '2026': 262},
 'filing_type_by_year': {'A': {2013: 448,
   2014: 306,
   2015: 213,
   2016: 230,
   2017: 256,
   2018: 239,
   2019: 202,
   2020: 161,
   2021: 164,
   2022: 180,
   2023: 129,
   2024: 101,
   2025: 85,
   2026: 28},
  'B': {2013: 1,
   2014: 0,
   2015: 0,
   2016: 0,
   2017: 0,
   2018: 1,
   2019: 0,
   2020: 0,
   2021: 3,
   2022: 6,
   2023: 7,
   2024: 4,
   2025: 2,
   2026: 0},
  'C': {2013: 15,
   2014: 776,
   2015: 401,
   2016: 715,
   2017: 1001,
   2018: 1075,
   2019: 964,
   2020: 780,
   2021: 753,
   2022: 861,
   2023: 621,
   2024: 661,
   2025: 891,
   2026: 574},
  'D': {2013: 92,
   2014: 224,
   2015: 130,
   2016: 164,
   2017

In [6]:
if not df_all.empty:
    display(pd.crosstab(df_all["year"], df_all["filing_type"]))

if not df_ptr.empty:
    display(df_ptr.groupby("year").size().rename("n_ptr"))

filing_type,A,B,C,D,E,G,H,O,P,R,T,W,X
year,,,,,,,,,,,,,
2013,448,1,15,92,5,4,0,1221,8,1,76,16,322
2014,306,0,776,224,1,5,50,396,708,0,5,109,208
2015,213,0,401,130,5,1,0,440,728,0,50,31,298
2016,230,0,715,164,0,1,45,393,765,0,5,87,299
2017,256,0,1001,336,7,1,0,437,801,0,49,91,509
2018,239,1,1075,228,2,22,87,355,830,0,6,279,319
2019,202,0,964,413,5,0,0,434,683,0,85,107,743
2020,161,0,780,164,2,0,62,374,733,0,4,83,567
2021,164,3,753,116,4,4,1,432,680,0,53,19,489


year
2013      8
2014    708
2015    728
2016    765
2017    801
2018    830
2019    683
2020    733
2021    680
2022    624
2023    460
2024    451
2025    515
2026    262
Name: n_ptr, dtype: int64

In [7]:
checks = {
    "missing_doc_id": int((df_all["doc_id"].astype(str).str.strip() == "").sum()) if not df_all.empty else 0,
    "missing_filing_date": int(df_all["filing_date"].isna().sum()) if not df_all.empty else 0,
    "duplicates_year_doc_id": int(df_all.duplicated(["year", "doc_id"]).sum()) if not df_all.empty else 0,
}
checks

{'missing_doc_id': 0, 'missing_filing_date': 699, 'duplicates_year_doc_id': 18}

In [8]:
output_paths = save_index_outputs(df_all, df_ptr, PROCESSED_HOUSE_DIR)
for name, path in output_paths.items():
    print(name, "->", Path(path).relative_to(ROOT))

house_filings_index -> data/processed/house/house_filings_index.csv
house_ptr_index -> data/processed/house/house_ptr_index.csv
ptr_index_2013 -> data/processed/house/ptr_index_2013.csv
ptr_index_2014 -> data/processed/house/ptr_index_2014.csv
ptr_index_2015 -> data/processed/house/ptr_index_2015.csv
ptr_index_2016 -> data/processed/house/ptr_index_2016.csv
ptr_index_2017 -> data/processed/house/ptr_index_2017.csv
ptr_index_2018 -> data/processed/house/ptr_index_2018.csv
ptr_index_2019 -> data/processed/house/ptr_index_2019.csv
ptr_index_2020 -> data/processed/house/ptr_index_2020.csv
ptr_index_2021 -> data/processed/house/ptr_index_2021.csv
ptr_index_2022 -> data/processed/house/ptr_index_2022.csv
ptr_index_2023 -> data/processed/house/ptr_index_2023.csv
ptr_index_2024 -> data/processed/house/ptr_index_2024.csv
ptr_index_2025 -> data/processed/house/ptr_index_2025.csv
ptr_index_2026 -> data/processed/house/ptr_index_2026.csv


In [9]:
logs_path = AUDIT_DIR / "house_index_download_logs.csv"
logs.to_csv(logs_path, index=False)
report_path = write_house_index_report(summary, output_paths)

print("Written:", logs_path.relative_to(ROOT))
print("Written:", report_path.relative_to(ROOT))

Written: data/audit/house_index_download_logs.csv
Written: reports/house_index_audit.md


## Conclusion

**OK** si `house_ptr_index.csv` existe et que les anomalies sont explicites.

**NEXT** — télécharger les PDF avec `02_house_pdf_download_manifest.ipynb`.